# W10: 공급업체 커뮤니케이션 자동화

이슈 유형별로 공급업체 공문을 생성하고, 파일별로 ZIP 묶음을 생성해 다운로드합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

import zipfile
from pathlib import Path

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    df = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8")))
else:
    df = pd.DataFrame(
        {
            "업체명": ["식자재1", "물류2", "패키지3"],
            "이슈유형": ["지연", "품질", "재고부족"],
            "세부내역": ["배송 지연", "파손", "재고 소진 임박"],
            "요청": ["즉시 조치", "대체물량", "우선 발주"]
        }
    )

if not set(["업체명", "이슈유형", "세부내역", "요청"]).issubset(df.columns):
    raise ValueError("필수 컬럼 누락")

Path("vendor_letters").mkdir(exist_ok=True)
for r in df.itertuples():
    prompt = f"[{r.업체명}] 이슈유형:{r.이슈유형}, 내용:{r.세부내역}, 요청:{r.요청} 공문형 메일 3문장 생성"
    txt = use_gemini(
        prompt,
        f"[{r.업체명}] 귀사에서 발생한 {r.이슈유형} 건에 대해 즉시 점검 후 대응 일정을 회신해 주시기 바랍니다. 재발 방지 대책 공유 바랍니다."
    )
    Path("vendor_letters").joinpath(f"{r.업체명}_{r.이슈유형}.txt").write_text(txt, encoding="utf-8")

with zipfile.ZipFile("vendor_communications.zip", "w") as zf:
    for file in Path("vendor_letters").glob("*.txt"):
        zf.write(file, arcname=file.name)

print("vendor_communications.zip 생성")
